In [ ]:
# Install required libraries
!pip install -q langchain langchain-groq langchain-community sentence-transformers faiss-cpu gradio ragas

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

# --- CONFIGURATION ---
GROQ_API_KEY = "YOUR_GROQ_API_KEY"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1
)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
# 1. Load Data (Example: LangChain Documentation)
loader = WebBaseLoader("https://python.langchain.com/docs/introduction/")
docs = loader.load()

In [ ]:
# 2. Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

In [ ]:

# 3. Vector Store
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

In [ ]:
# 4. RAG Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# 5. Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# --- GRADIO INTERFACE ---
def chat_function(message, history):
    return rag_chain.invoke(message)

demo = gr.ChatInterface(
    fn=chat_function, 
    title="Llama 3.1 Naive RAG (Groq)",
    description="Ask questions about the LangChain introduction documentation."
)

In [ ]:
if __name__ == "__main__":
    demo.launch(debug=True)